#Importing relevant modules

In [1]:
from importlib import import_module

In [8]:
detector_module = import_module("legal-ai.rules.detector")

In [10]:
engine_module = import_module("legal-ai.rules.engine")

engine_module = import_module("legal-ai.rules.engine")

In [11]:
ConcerRule = engine_module.ConcernRule
Detector = detector_module.ConcernDetector
RuleStore = engine_module.InMemoryRuleStore

In [14]:
print(engine_module.load_rules()[0])

ConcernRule(rule_id='demo.non_compete.duration_over_12_months', clause_type='non_compete', field='duration', operator='greater_than', finding_type='duration_review', severity='high', description='Review non-compete durations exceeding the configured 12-month threshold.', source='demo_configuration', value=12, unit='month', exception_field='exceptions')


### First test is without the extraction layer

In [25]:
clause_text = "Employee agrees not to compete for twenty-four (24) months, except with the prior written consent of the Board of Directors."
extracted = {
  "duration": "twenty-four (24) months",
  "geographical_scope": "California",
  "exceptions": ["prior written consent of the Board of Directors"]
}



In [19]:
detector = Detector()

In [26]:
result = detector.detect(
    clause_type="non_compete",
    extraction_output=extracted,
    clause_text=clause_text
)

In [27]:
import json
print(json.dumps(result.to_dict(), indent=4, ensure_ascii=False))

{
    "findings": [],
    "retrieved_rule_ids": [
        "demo.non_compete.duration_over_12_months",
        "demo.non_compete.worldwide_scope"
    ]
}


### Now i try it with the extraction layer

In [28]:
# Rebuilt Strict Gold Dataset (50 Clauses with 100% Explicit Ground Truth)
BENCHMARK_DATASET = [
    # --- 1. confidentiality ---
    {
        "id": "CONF_001", "category": "confidentiality",
        "text": "The Recipient agrees to hold in strict confidence all proprietary technical data and trade secrets disclosed by the Disclosing Party, and shall not disclose such information for a period of five (5) years.",
        "expected": {"protected_information": ["proprietary technical data", "trade secrets"], "obligations": ["hold in strict confidence", "shall not disclose"], "duration": "five (5) years", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_002", "category": "confidentiality",
        "text": "Executive shall keep all customer lists and financial records strictly confidential. This obligation survives termination of employment indefinitely.",
        "expected": {"protected_information": ["customer lists", "financial records"], "obligations": ["keep all customer lists and financial records strictly confidential"], "duration": None, "post_termination": "survives termination of employment indefinitely", "exceptions": []}
    },
    {
        "id": "CONF_003", "category": "confidentiality",
        "text": "Consultant will protect all source code and technical data from unauthorized disclosure during the engagement and for three (3) years thereafter.",
        "expected": {"protected_information": ["source code", "technical data"], "obligations": ["protect all source code and technical data from unauthorized disclosure"], "duration": "three (3) years", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_004", "category": "confidentiality",
        "text": "All proprietary technical data disclosed hereunder shall remain confidential for twelve (12) months following contract completion.",
        "expected": {"protected_information": ["proprietary technical data"], "obligations": ["shall remain confidential"], "duration": "twelve (12) months", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_005", "category": "confidentiality",
        "text": "Receiving Party shall safeguard trade secrets and financial records, provided that information already in public domain is excluded.",
        "expected": {"protected_information": ["trade secrets", "financial records"], "obligations": ["shall safeguard"], "duration": None, "post_termination": None, "exceptions": ["information already in public domain"]}
    },

    # --- 2. exclusivity ---
    {
        "id": "EXCL_001", "category": "exclusivity",
        "text": "During the term of this Agreement, the Consultant shall work exclusively for the Client and shall not perform similar consulting services for any third party without prior written consent.",
        "expected": {"exclusive_to": "Client", "restricted_activities": ["perform similar consulting services for any third party"], "duration": "During the term of this Agreement", "exceptions": ["prior written consent"]}
    },
    {
        "id": "EXCL_002", "category": "exclusivity",
        "text": "Employee covenants to devote full professional time exclusively to Employer for a period of two (2) years.",
        "expected": {"exclusive_to": "Employer", "restricted_activities": ["devote full professional time exclusively to Employer"], "duration": "two (2) years", "exceptions": []}
    },
    {
        "id": "EXCL_003", "category": "exclusivity",
        "text": "Distributor shall act as the sole and exclusive distributor of the Products in the Territory during the five (5) year term.",
        "expected": {"exclusive_to": None, "restricted_activities": ["act as the sole and exclusive distributor of the Products in the Territory"], "duration": "five (5) year", "exceptions": []}
    },
    {
        "id": "EXCL_004", "category": "exclusivity",
        "text": "Contractor agrees to provide services exclusively to the Company throughout the engagement.",
        "expected": {"exclusive_to": "Company", "restricted_activities": ["provide services exclusively to the Company"], "duration": "throughout the engagement", "exceptions": []}
    },
    {
        "id": "EXCL_005", "category": "exclusivity",
        "text": "Agent shall not represent any competing principal during the twelve (12) month contract period.",
        "expected": {"exclusive_to": None, "restricted_activities": ["represent any competing principal"], "duration": "twelve (12) month", "exceptions": []}
    },

    # --- 3. governing_law ---
    {
        "id": "GOV_001", "category": "governing_law",
        "text": "This Agreement shall be governed by, construed, and enforced in accordance with the internal laws of the State of Delaware, without regard to its conflict of laws principles.",
        "expected": {"jurisdiction": "State of Delaware", "governing_law": "internal laws of the State of Delaware"}
    },
    {
        "id": "GOV_002", "category": "governing_law",
        "text": "All disputes arising under this contract shall be submitted to the exclusive jurisdiction of the state and federal courts in New York.",
        "expected": {"jurisdiction": "state and federal courts in New York", "governing_law": None}
    },
    {
        "id": "GOV_003", "category": "governing_law",
        "text": "This Agreement is governed by the laws of the State of California.",
        "expected": {"jurisdiction": None, "governing_law": "laws of the State of California"}
    },
    {
        "id": "GOV_004", "category": "governing_law",
        "text": "The validity and interpretation of this contract shall be governed by the laws of England and Wales.",
        "expected": {"jurisdiction": None, "governing_law": "laws of England and Wales"}
    },
    {
        "id": "GOV_005", "category": "governing_law",
        "text": "This Agreement shall be interpreted in accordance with the laws of Delaware.",
        "expected": {"jurisdiction": None, "governing_law": "laws of Delaware"}
    },

    # --- 4. ip_assignment ---
    {
        "id": "IP_001", "category": "ip_assignment",
        "text": "Employee hereby irrevocably assigns and transfers to Employer all right, title, and interest in and to all inventions, code, and patents created during the period of employment.",
        "expected": {"assignor": "Employee", "assignee": "Employer", "assigned_rights": ["inventions", "code", "patents"], "scope": "created during the period of employment", "exceptions": []}
    },
    {
        "id": "IP_002", "category": "ip_assignment",
        "text": "Consultant assigns to Company all copyrights, trademarks, and trade secrets developed under this contract.",
        "expected": {"assignor": "Consultant", "assignee": "Company", "assigned_rights": ["copyrights", "trademarks", "trade secrets"], "scope": "developed under this contract", "exceptions": []}
    },
    {
        "id": "IP_003", "category": "ip_assignment",
        "text": "Contractor hereby assigns all inventions and works of authorship created during performance of services to Company.",
        "expected": {"assignor": "Contractor", "assignee": "Company", "assigned_rights": ["inventions", "works of authorship"], "scope": "created during performance of services", "exceptions": []}
    },
    {
        "id": "IP_004", "category": "ip_assignment",
        "text": "Employee assigns all patents and code developed during employment to Employer.",
        "expected": {"assignor": "Employee", "assignee": "Employer", "assigned_rights": ["patents", "code"], "scope": "developed during employment", "exceptions": []}
    },
    {
        "id": "IP_005", "category": "ip_assignment",
        "text": "Consultant assigns all right, title, and interest in inventions and copyrights to Company.",
        "expected": {"assignor": "Consultant", "assignee": "Company", "assigned_rights": ["inventions", "copyrights"], "scope": None, "exceptions": []}
    },

    # --- 5. liability_indemnity ---
    {
        "id": "IND_001", "category": "liability_indemnity",
        "text": "The Contractor agrees to defend, indemnify, and hold harmless the Company against all third-party claims, liabilities, and legal damages arising from negligence.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims", "liabilities", "negligence"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_002", "category": "liability_indemnity",
        "text": "Service Provider shall indemnify Client against all losses, damages, and attorney fees resulting from copyright infringement.",
        "expected": {"indemnifying_party": "Service Provider", "protected_party": "Client", "covered_claims": ["copyright infringement"], "covered_damages": ["losses", "damages", "attorney fees"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_003", "category": "liability_indemnity",
        "text": "Contractor agrees to indemnify Company for all third-party claims and legal damages.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_004", "category": "liability_indemnity",
        "text": "Service Provider shall defend Client against infringement claims and pay all legal damages.",
        "expected": {"indemnifying_party": "Service Provider", "protected_party": "Client", "covered_claims": ["infringement claims"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_005", "category": "liability_indemnity",
        "text": "Contractor shall hold harmless Company against third-party claims arising from gross negligence.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims", "gross negligence"], "covered_damages": [], "limitations": [], "exceptions": []}
    },

    # --- 6. no_solicit_customers ---
    {
        "id": "NSC_001", "category": "no_solicit_customers",
        "text": "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit or approach any customer or client of the Company.",
        "expected": {"restricted_action": "solicit or approach", "restricted_entities": ["customer", "client of the Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "termination of employment", "exceptions": []}
    },
    {
        "id": "NSC_002", "category": "no_solicit_customers",
        "text": "During employment and for twenty-four (24) months thereafter, Executive shall not solicit active accounts or customers of Employer.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["active accounts", "customers of Employer"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "thereafter", "exceptions": []}
    },
    {
        "id": "NSC_003", "category": "no_solicit_customers",
        "text": "Contractor agrees not to solicit any client of Company for eighteen (18) months post-separation.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["client of Company"], "duration": "eighteen (18) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },
    {
        "id": "NSC_004", "category": "no_solicit_customers",
        "text": "Employee shall not solicit customer accounts of Employer for twelve (12) months after termination.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["customer accounts of Employer"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "after termination", "exceptions": []}
    },
    {
        "id": "NSC_005", "category": "no_solicit_customers",
        "text": "Executive agrees not to solicit or approach clients of Company for one (1) year post-separation.",
        "expected": {"restricted_action": "solicit or approach", "restricted_entities": ["clients of Company"], "duration": "one (1) year", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },

    # --- 7. no_solicit_employees ---
    {
        "id": "NSE_001", "category": "no_solicit_employees",
        "text": "For a period of twenty-four (24) months post-separation, the Executive will not induce or entice away any senior engineers or personnel of the Employer.",
        "expected": {"restricted_action": "induce or entice away", "restricted_entities": ["senior engineers", "personnel of the Employer"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },
    {
        "id": "NSE_002", "category": "no_solicit_employees",
        "text": "Employee shall not solicit or hire any employees or contractors of Company for twelve (12) months following termination.",
        "expected": {"restricted_action": "solicit or hire", "restricted_entities": ["employees", "contractors of Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "following termination", "exceptions": []}
    },
    {
        "id": "NSE_003", "category": "no_solicit_employees",
        "text": "Consultant agrees not to entice away any personnel of Employer for eighteen (18) months after cessation of service.",
        "expected": {"restricted_action": "entice away", "restricted_entities": ["personnel of Employer"], "duration": "eighteen (18) months", "geographical_scope": None, "trigger": "after cessation of service", "exceptions": []}
    },
    {
        "id": "NSE_004", "category": "no_solicit_employees",
        "text": "Executive covenants not to solicit any key employees of Company for twenty-four (24) months post-termination.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["key employees of Company"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "post-termination", "exceptions": []}
    },
    {
        "id": "NSE_005", "category": "no_solicit_employees",
        "text": "Contractor agrees not to induce any employees or engineers of Company to terminate employment during twelve (12) months post-separation.",
        "expected": {"restricted_action": "induce", "restricted_entities": ["employees", "engineers of Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },

    # --- 8. non_compete ---
    {
        "id": "NC_001", "category": "non_compete",
        "text": "During employment and for eighteen (18) months thereafter, the Employee shall not operate or work for any competing business within North America.",
        "expected": {"restricted_activity": "operate or work for any competing business", "competitors": ["competing business"], "duration": "eighteen (18) months", "geographical_scope": "North America", "trigger": "thereafter", "exceptions": []}
    },
    {
        "id": "NC_002", "category": "non_compete",
        "text": "Executive agrees not to engage in competing business activities worldwide for twenty-four (24) months post-termination.",
        "expected": {"restricted_activity": "engage in competing business activities", "competitors": ["competing business"], "duration": "twenty-four (24) months", "geographical_scope": "worldwide", "trigger": "post-termination", "exceptions": []}
    },
    {
        "id": "NC_003", "category": "non_compete",
        "text": "Consultant shall not work for any direct competitors in the United States for twelve (12) months following separation.",
        "expected": {"restricted_activity": "work for any direct competitors", "competitors": ["direct competitors"], "duration": "twelve (12) months", "geographical_scope": "United States", "trigger": "following separation", "exceptions": []}
    },
    {
        "id": "NC_004", "category": "non_compete",
        "text": "Employee will not engage in competing business activities within North America for eighteen (18) months after termination.",
        "expected": {"restricted_activity": "engage in competing business activities", "competitors": ["competing business"], "duration": "eighteen (18) months", "geographical_scope": "North America", "trigger": "after termination", "exceptions": []}
    },
    {
        "id": "NC_005", "category": "non_compete",
        "text": "Executive covenants not to operate any competing business worldwide for two (2) years post-separation.",
        "expected": {"restricted_activity": "operate any competing business", "competitors": ["competing business"], "duration": "two (2) years", "geographical_scope": "worldwide", "trigger": "post-separation", "exceptions": []}
    },

    # --- 9. other_general ---
    {
        "id": "GEN_001", "category": "other_general",
        "text": "All notices required under this Agreement shall be delivered in writing via certified mail or email to the designated corporate headquarters.",
        "expected": {"obligations": ["delivered in writing via certified mail or email to the designated corporate headquarters"], "rights": [], "conditions": [], "entities": []}
    },
    {
        "id": "GEN_002", "category": "other_general",
        "text": "Company reserves the right to audit books and records of Contractor upon reasonable written notice.",
        "expected": {"obligations": [], "rights": ["audit books and records of Contractor"], "conditions": ["upon reasonable written notice"], "entities": ["Company", "Contractor"]}
    },
    {
        "id": "GEN_003", "category": "other_general",
        "text": "Contractor shall comply with security protocols and deliver notices in writing.",
        "expected": {"obligations": ["comply with security protocols", "deliver notices in writing"], "rights": [], "conditions": [], "entities": ["Contractor"]}
    },
    {
        "id": "GEN_004", "category": "other_general",
        "text": "Auditor has the right to inspect facilities and audit books and records.",
        "expected": {"obligations": [], "rights": ["inspect facilities", "audit books and records"], "conditions": [], "entities": ["Auditor"]}
    },
    {
        "id": "GEN_005", "category": "other_general",
        "text": "All notices shall be delivered in writing to Company headquarters.",
        "expected": {"obligations": ["delivered in writing to Company headquarters"], "rights": [], "conditions": [], "entities": ["Company"]}
    },

    # --- 10. termination_notice ---
    {
        "id": "TERM_001", "category": "termination_notice",
        "text": "Either party may terminate this Agreement without cause by giving at least thirty (30) days prior written notice to the other party.",
        "expected": {"notice_period": "thirty (30) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["giving at least thirty (30) days prior written notice"], "effective_date": None}
    },
    {
        "id": "TERM_002", "category": "termination_notice",
        "text": "Employer may terminate employment immediately for cause upon written notice to Employee.",
        "expected": {"notice_period": "immediately", "notice_given_by": "Employer", "termination_type": "for cause", "conditions": ["upon written notice"], "effective_date": None}
    },
    {
        "id": "TERM_003", "category": "termination_notice",
        "text": "Either party may terminate this contract without cause upon sixty (60) days notice.",
        "expected": {"notice_period": "sixty (60) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["upon sixty (60) days notice"], "effective_date": None}
    },
    {
        "id": "TERM_004", "category": "termination_notice",
        "text": "Company may terminate this Agreement for cause upon thirty (30) days written notice.",
        "expected": {"notice_period": "thirty (30) days", "notice_given_by": "Company", "termination_type": "for cause", "conditions": ["upon thirty (30) days written notice"], "effective_date": None}
    },
    {
        "id": "TERM_005", "category": "termination_notice",
        "text": "Either party may terminate without cause by providing ninety (90) days notice.",
        "expected": {"notice_period": "ninety (90) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["providing ninety (90) days notice"], "effective_date": None}
    }
]

print(f"[✓] Loaded strict gold dataset with {len(BENCHMARK_DATASET)} ground-truth clauses.")

[✓] Loaded strict gold dataset with 50 ground-truth clauses.


In [30]:
import os
import sys
import re
import json
import warnings
import pandas as pd
import torch
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

In [31]:
# General Extraction Rules for NuExtract 2.0 Prompting
GENERAL_EXTRACTION_RULES = """
Extract structured information from the text according to the provided template.

RULES:
- Extract values only from the provided text.
- Never copy or output the field description as a value.
- Never invent, infer, or assume information that is not explicitly stated.
- Preserve the original wording from the text whenever possible.
- Extract only the span relevant to each field.
- Do not include information belonging to another field.
- Split coordinated lists into separate items when the text clearly lists multiple items.
- Keep related phrases together when they form one semantic entity.
- For action fields, extract only the prohibited/required action, not the subject, object, or time period.
- For entity fields, extract only the entity or entity phrase.
- Put temporal information in the appropriate duration, trigger, scope, or notice_period field.
- Do not put temporal information inside assigned_rights, restricted_action, restricted_entities, or similar fields.
- If a field is not explicitly stated in the text, return null for scalar fields and [] for list fields.
- Do not paraphrase the text unless necessary to make the extracted value grammatical.
- Do not change the legal meaning of an extracted value.
- Distinguish between fixed 'duration' (e.g., "five (5) years") vs 'post_termination' / 'trigger' survival timing (e.g., "post-separation", "thereafter", "following termination").
- For IP assignment, extract situational or temporal constraints into 'scope' (e.g., "created during employment", "developed under this contract").
- For non-solicitation, extract target subjects into 'restricted_entities' (e.g., "customer accounts of Employer", "senior engineers", "personnel of Employer").
- For general clauses, extract stated requirements into 'obligations', rights granted into 'rights', prerequisites into 'conditions', and referenced parties into 'entities'.
"""

# Targeted Few-Shot Prompting Examples for ALL 10 Categories in NuExtract 2.0
CATEGORY_FEW_SHOT_EXAMPLES: Dict[str, str] = {
    "confidentiality": """Example Text: "Consultant will protect all source code and technical data from unauthorized disclosure during the engagement and for three (3) years thereafter."
Example Output:
{
  "protected_information": ["source code", "technical data"],
  "obligations": ["protect all source code and technical data from unauthorized disclosure"],
  "duration": "three (3) years",
  "post_termination": null,
  "exceptions": []
}""",
    "exclusivity": """Example Text: "During the term of this Agreement, the Consultant shall work exclusively for the Client and shall not perform similar consulting services for any third party without prior written consent."
Example Output:
{
  "exclusive_to": "Client",
  "restricted_activities": ["perform similar consulting services for any third party"],
  "duration": "During the term of this Agreement",
  "exceptions": ["prior written consent"]
}""",
    "governing_law": """Example Text: "This Agreement shall be governed by, construed, and enforced in accordance with the internal laws of the State of Delaware, without regard to its conflict of laws principles."
Example Output:
{
  "jurisdiction": "State of Delaware",
  "governing_law": "internal laws of the State of Delaware"
}""",
    "ip_assignment": """Example Text: "Employee hereby irrevocably assigns and transfers to Employer all right, title, and interest in and to all inventions, code, and patents created during the period of employment."
Example Output:
{
  "assignor": "Employee",
  "assignee": "Employer",
  "assigned_rights": ["inventions", "code", "patents"],
  "scope": "created during the period of employment",
  "exceptions": []
}""",
    "liability_indemnity": """Example Text: "The Contractor agrees to defend, indemnify, and hold harmless the Company against all third-party claims, liabilities, and legal damages arising from negligence."
Example Output:
{
  "indemnifying_party": "Contractor",
  "protected_party": "Company",
  "covered_claims": ["third-party claims", "liabilities", "negligence"],
  "covered_damages": ["legal damages"],
  "limitations": [],
  "exceptions": []
}""",
    "no_solicit_customers": """Example Text: "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit or approach any customer or client of the Company."
Example Output:
{
  "restricted_action": "solicit or approach",
  "restricted_entities": ["customer", "client of the Company"],
  "duration": "twelve (12) months",
  "geographical_scope": null,
  "trigger": "termination of employment",
  "exceptions": []
}""",
    "no_solicit_employees": """Example Text: "For a period of twenty-four (24) months post-separation, the Executive will not induce or entice away any senior engineers or personnel of the Employer."
Example Output:
{
  "restricted_action": "induce or entice away",
  "restricted_entities": ["senior engineers", "personnel of the Employer"],
  "duration": "twenty-four (24) months",
  "geographical_scope": null,
  "trigger": "post-separation",
  "exceptions": []
}""",
    "non_compete": """Example Text: "During employment and for eighteen (18) months thereafter, the Employee shall not operate or work for any competing business within North America."
Example Output:
{
  "restricted_activity": "operate or work for any competing business",
  "competitors": ["competing business"],
  "duration": "eighteen (18) months",
  "geographical_scope": "North America",
  "trigger": "thereafter",
  "exceptions": []
}""",
    "other_general": """Example Text: "Contractor shall comply with security protocols and deliver notices in writing."
Example Output:
{
  "obligations": ["comply with security protocols", "deliver notices in writing"],
  "rights": [],
  "conditions": [],
  "entities": ["Contractor"]
}""",
    "termination_notice": """Example Text: "Either party may terminate this Agreement without cause by giving at least thirty (30) days prior written notice to the other party."
Example Output:
{
  "notice_period": "thirty (30) days",
  "notice_given_by": "Either party",
  "termination_type": "without cause",
  "conditions": ["giving at least thirty (30) days prior written notice"],
  "effective_date": null
}"""
}

def format_nuextract_prompt(text: str, schema: Any, category: Optional[str] = None, local_context: Optional[str] = None) -> str:
    """Constructs the formatted NuExtract 2.0 prompt string with targeted Few-Shot In-Context Examples."""
    full_text = f"{local_context}\n\n{text}" if local_context else text
    schema_json = json.dumps(schema, indent=2) if isinstance(schema, dict) else str(schema)
    
    few_shot_block = ""
    if category and category in CATEGORY_FEW_SHOT_EXAMPLES:
        few_shot_block = f"\n### Few-Shot Example:\n{CATEGORY_FEW_SHOT_EXAMPLES[category]}\n"
        
    return f"""<|input|>
### Instructions:
{GENERAL_EXTRACTION_RULES}

### Template:
{schema_json}
{few_shot_block}
### Text:
{full_text}

<|output|>
"""

# Category -> Family Mapping
CATEGORY_TO_FAMILY: Dict[str, str] = {
    "confidentiality": "confidentiality_ip",
    "ip_assignment": "confidentiality_ip",
    "exclusivity": "restrictive_covenant",
    "no_solicit_customers": "restrictive_covenant",
    "no_solicit_employees": "restrictive_covenant",
    "non_compete": "restrictive_covenant",
    "governing_law": "legal_contractual",
    "liability_indemnity": "legal_contractual",
    "termination_notice": "legal_contractual",
    "other_general": "general"
}

# Default Blank Schemas
EXTRACTION_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "confidentiality": {"protected_information": [], "obligations": [], "duration": None, "post_termination": None, "exceptions": []},
    "exclusivity": {"exclusive_to": None, "restricted_activities": [], "duration": None, "exceptions": []},
    "governing_law": {"jurisdiction": None, "governing_law": None},
    "ip_assignment": {"assignor": None, "assignee": None, "assigned_rights": [], "scope": None, "exceptions": []},
    "liability_indemnity": {"indemnifying_party": None, "protected_party": None, "covered_claims": [], "covered_damages": [], "limitations": [], "exceptions": []},
    "no_solicit_customers": {"restricted_action": None, "restricted_entities": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "no_solicit_employees": {"restricted_action": None, "restricted_entities": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "non_compete": {"restricted_activity": None, "competitors": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "other_general": {"obligations": [], "rights": [], "conditions": [], "entities": []},
    "termination_notice": {"notice_period": None, "notice_given_by": None, "termination_type": None, "conditions": [], "effective_date": None}
}

# Enriched Schemas with Explicit Semantic Field Guidance for NuExtract Prompting
EXTRACTION_SCHEMAS_WITH_DESCRIPTIONS: Dict[str, Dict[str, Any]] = {
    "confidentiality": {
        "protected_information": "List of specific types of information protected",
        "obligations": "List of confidentiality duties stated in clause",
        "duration": "Duration of confidentiality obligation",
        "post_termination": "Post-termination survival obligation if explicitly stated",
        "exceptions": "List of explicit exceptions"
    },
    "exclusivity": {
        "exclusive_to": "Entity receiving exclusive services/rights",
        "restricted_activities": "List of restricted outside activities",
        "duration": "Duration of exclusivity",
        "exceptions": "List of explicit exceptions"
    },
    "governing_law": {
        "jurisdiction": "State, venue, or court jurisdiction",
        "governing_law": "Law governing the agreement"
    },
    "ip_assignment": {
        "assignor": "Party giving up/assigning IP rights",
        "assignee": "Party receiving assigned IP rights",
        "assigned_rights": "List of specific IP rights assigned",
        "scope": "Situational or temporal scope",
        "exceptions": "List of explicit IP exceptions"
    },
    "liability_indemnity": {
        "indemnifying_party": "Party obligated to defend or indemnify",
        "protected_party": "Party receiving indemnity protection",
        "covered_claims": "List of covered claims or triggers",
        "covered_damages": "List of covered financial losses/damages",
        "limitations": "List of liability limits",
        "exceptions": "List of explicit exceptions"
    },
    "no_solicit_customers": {
        "restricted_action": "Specific restricted non-solicitation behavior",
        "restricted_entities": "List of protected customer/client entities",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area if specified",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "no_solicit_employees": {
        "restricted_action": "Specific restricted non-solicitation behavior",
        "restricted_entities": "List of protected employee/personnel entities",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area if specified",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "non_compete": {
        "restricted_activity": "Description of restricted competitive activity",
        "competitors": "List of competitor entities or firms",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area of restriction",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "other_general": {
        "obligations": "List of explicit general duties or obligations stated in clause",
        "rights": "List of explicit general rights granted in clause",
        "conditions": "List of stated conditions or prerequisites",
        "entities": "List of entities mentioned"
    },
    "termination_notice": {
        "notice_period": "Required notice duration",
        "notice_given_by": "Party entitled or required to give notice",
        "termination_type": "Type of termination",
        "conditions": "List of termination conditions",
        "effective_date": "Effective date if specified"
    }
}

print(f"[✓] Loaded GENERAL_EXTRACTION_RULES, Few-Shot examples ({len(CATEGORY_FEW_SHOT_EXAMPLES)} categories), and schema prompts.")

[✓] Loaded GENERAL_EXTRACTION_RULES, Few-Shot examples (10 categories), and schema prompts.


## End-to-End Pipeline: Extraction + Rule Review Evaluation

This section tests the full pipeline on the **50 synthetic gold-standard clauses** (`BENCHMARK_DATASET`):
1. **Ground Truth Evaluation**: Evaluates rules on the *expected* gold facts to establish the **expected findings**.
2. **Information Extraction**: Runs `NuExtractExtractor` to extract facts from the raw clause text.
3. **Extracted Facts Evaluation**: Evaluates rules on the *actually extracted* facts to produce the **actual findings**.
4. **Manual & Side-by-Side Review**: Compares expected vs extracted entities and expected vs actual rule findings.

In [ ]:
from importlib import import_module
import json

# 1. Load Extractor & Detector modules
extractor_module = import_module("legal-ai.extraction.llm_extractor")
detector_module = import_module("legal-ai.rules.detector")
NuExtractExtractor = extractor_module.NuExtractExtractor
ConcernDetector = detector_module.ConcernDetector

# By default use_fallback=True runs the fast local heuristic extractor.
# Set use_fallback=False if you want to run the full HuggingFace 2B model weights.
extractor = NuExtractExtractor(use_fallback=True)
detector = ConcernDetector()

def run_pipeline_on_dataset(dataset, extractor, detector):
    """Run extraction and rule-review detection on a dataset of clauses."""
    records = []
    for item in dataset:
        clause_id = item.get("id", "UNKNOWN")
        category = item["category"]
        clause_text = item["text"]
        expected = item.get("expected", {})
        
        # 1. Expected Findings (from ground-truth facts)
        expected_result = detector.detect(
            clause_type=category,
            extraction_output=expected,
            clause_text=clause_text
        )
        expected_findings = [f.to_dict() for f in expected_result.findings]
        
        # 2. Information Extraction (from raw clause text)
        raw_output = extractor.extract(clause_text, category)
        try:
            extracted = json.loads(raw_output) if isinstance(raw_output, str) else raw_output
        except Exception as e:
            extracted = {"error": str(e), "raw": raw_output}
            
        # 3. Actually Detected Findings (from extracted facts)
        actual_result = detector.detect(
            clause_type=category,
            extraction_output=extracted,
            clause_text=clause_text
        )
        actual_findings = [f.to_dict() for f in actual_result.findings]
        
        # Check if rule findings match
        exp_ids = sorted([f["rule_id"] for f in expected_findings])
        act_ids = sorted([f["rule_id"] for f in actual_findings])
        findings_match = (exp_ids == act_ids)
        
        records.append({
            "id": clause_id,
            "category": category,
            "text": clause_text,
            "expected": expected,
            "extracted": extracted,
            "expected_findings": expected_findings,
            "actual_findings": actual_findings,
            "findings_match": findings_match,
            "retrieved_rule_ids": actual_result.retrieved_rule_ids
        })
    return records

def _format_findings_list(findings):
    if not findings:
        return "  None (Clean / No concerns triggered)\n"
    lines = []
    for i, f in enumerate(findings, 1):
        lines.append(f"  {i}. [{f['severity'].upper()}] Rule: {f['rule_id']}")
        lines.append(f"     Field    : {f['field']} = {f['extracted_value']}")
        lines.append(f"     Condition: {f['rule_condition']}")
        if f.get('evidence'):
            lines.append(f"     Evidence : {f['evidence']}")
    return "\n".join(lines) + "\n"

def display_result(record):
    """Clean, comprehensive side-by-side display of a single clause's output."""
    match_indicator = "✓ MATCH" if record["findings_match"] else "✗ MISMATCH"
    print("=" * 90)
    print(f"[{record['id']}] CATEGORY: {record['category']}  |  FINDINGS STATUS: [{match_indicator}]")
    print("-" * 90)
    print(f"CLAUSE TEXT:\n{record['text']}\n")
    print(f"EXPECTED EXTRACTION:\n{json.dumps(record['expected'], indent=2, ensure_ascii=False)}")
    print(f"\nACTUALLY EXTRACTED:\n{json.dumps(record['extracted'], indent=2, ensure_ascii=False)}")
    print("-" * 90)
    print(f"EXPECTED FINDINGS (from gold facts):")
    print(_format_findings_list(record['expected_findings']), end="")
    print(f"ACTUALLY DETECTED FINDINGS (from extracted facts):")
    print(_format_findings_list(record['actual_findings']), end="")
    print("=" * 90 + "\n")

print("[✓] Pipeline runner and display functions defined successfully.")

In [ ]:
# Run pipeline across all 50 clauses in BENCHMARK_DATASET
pipeline_results = run_pipeline_on_dataset(BENCHMARK_DATASET, extractor, detector)

total = len(pipeline_results)
matches = sum(1 for r in pipeline_results if r['findings_match'])
flagged = sum(1 for r in pipeline_results if r['actual_findings'])
clean = total - flagged

print("=" * 65)
print(f"PIPELINE BENCHMARK SUMMARY (50 CLAUSES):")
print(f"- Total Clauses Evaluated          : {total}")
print(f"- Clauses with Concerns Triggered  : {flagged} ({flagged/total:.1%})")
print(f"- Clean Clauses (No Concerns)      : {clean} ({clean/total:.1%})")
print(f"- Rule Findings Match (Expected=Act): {matches} / {total} ({matches/total:.1%})")
print("=" * 65)

In [ ]:
# Manual review: Print clean output for each of the 50 clauses
for record in pipeline_results:
    display_result(record)

In [ ]:
# Helper: Quickly inspect a specific clause by ID or category
def inspect_clause(clause_id_or_index):
    if isinstance(clause_id_or_index, int):
        display_result(pipeline_results[clause_id_or_index])
    else:
        matches = [r for r in pipeline_results if r['id'] == clause_id_or_index]
        if matches:
            display_result(matches[0])
        else:
            print(f"Clause '{clause_id_or_index}' not found.")

# Example usage:
# inspect_clause("CONF_002")
# inspect_clause(0)